# 04 - DVC Pipelines

**Goal:** Define reproducible ML pipelines as code, so anyone can rerun the full workflow with one command.

---

## The Problem

To train a model, you need to run multiple steps in order:

```
1. Prepare data    (read CSVs, create splits)
2. Train model     (run training loop)
3. Evaluate model  (compute Dice on validation set)
```

Currently this is done manually. If someone changes the data or config, do they need to rerun everything? Which steps? DVC pipelines answer this automatically.

```
Manual:      "Hey, I changed the split CSV. Did you retrain?"
DVC:         dvc repro  →  automatically reruns what changed
```

## What is a DVC Pipeline?

A `dvc.yaml` file that describes your workflow as a **DAG** (directed acyclic graph):

```
prepare_data ──→ train ──→ evaluate
     │                        │
     ├── deps: raw_data/      ├── deps: model.pth
     ├── deps: config.yaml    ├── outs: metrics.json
     └── outs: splits/        └── outs: predictions/
```

Each stage has:
- **deps** (dependencies) — inputs that trigger rerun if changed
- **outs** (outputs) — files produced by the stage
- **cmd** — the command to run
- **params** — config values to track
- **metrics** — results to compare

## Setup

We'll build a mini pipeline in `/tmp/dvc-pipeline-demo`.

In [ ]:
import os
import shutil

DEMO_DIR = '/tmp/dvc-pipeline-demo'
if os.path.exists(DEMO_DIR):
    shutil.rmtree(DEMO_DIR)
os.makedirs(DEMO_DIR)
os.chdir(DEMO_DIR)

# Initialize git + DVC
!git init
!git branch -m main
!dvc init
!git add .
!git commit -m "Initialize project"

print(f"Working in: {os.getcwd()}")

## Create the Pipeline Scripts

We need three Python scripts — one per stage. These mirror the production training pipeline:

| Script | Production equivalent |
|--------|----------------------|
| `prepare_data.py` | `dataset.py` + CSV splits |
| `train.py` | `training_semantic.py` |
| `evaluate.py` | `testing_semantic.py` |

In [ ]:
%%writefile prepare_data.py
"""Stage 1: Prepare data — read config, create train/val splits."""
import json
import numpy as np
import os
import yaml

# Read config
with open("params.yaml") as f:
    config = yaml.safe_load(f)

num_samples = config["data"]["num_samples"]
val_ratio = config["data"]["val_ratio"]
seed = config.get("seed", 42)

np.random.seed(seed)

# Generate synthetic data (mimicking spine CT volumes)
os.makedirs("data/processed", exist_ok=True)

images = np.random.randn(num_samples, 1, 32, 32).astype(np.float32)
labels = np.random.randint(0, config["model"]["num_classes"], 
                           size=(num_samples, 32, 32)).astype(np.int64)

# Split
n_val = int(num_samples * val_ratio)
n_train = num_samples - n_val

np.save("data/processed/train_images.npy", images[:n_train])
np.save("data/processed/train_labels.npy", labels[:n_train])
np.save("data/processed/val_images.npy", images[n_train:])
np.save("data/processed/val_labels.npy", labels[n_train:])

# Save split info
split_info = {"n_train": n_train, "n_val": n_val, "seed": seed}
with open("data/processed/split_info.json", "w") as f:
    json.dump(split_info, f)

print(f"Prepared {n_train} train + {n_val} val samples")

In [ ]:
%%writefile train.py
"""Stage 2: Train model."""
import json
import numpy as np
import torch
import torch.nn as nn
import yaml
import os

# Read config
with open("params.yaml") as f:
    config = yaml.safe_load(f)

# Load data
train_images = torch.from_numpy(np.load("data/processed/train_images.npy"))
train_labels = torch.from_numpy(np.load("data/processed/train_labels.npy"))

# Simple model
num_classes = config["model"]["num_classes"]
features = config["model"]["features"]

model = nn.Sequential(
    nn.Conv2d(1, features, 3, padding=1), nn.ReLU(),
    nn.Conv2d(features, features * 2, 3, padding=1), nn.ReLU(),
    nn.Conv2d(features * 2, features, 3, padding=1), nn.ReLU(),
    nn.Conv2d(features, num_classes, 1),
)

optimizer = torch.optim.Adam(model.parameters(), lr=config["training"]["lr"])
criterion = nn.CrossEntropyLoss()

# Training loop
num_epochs = config["training"]["num_epochs"]
batch_size = config["training"]["batch_size"]
losses = []

model.train()
for epoch in range(num_epochs):
    epoch_loss = 0
    n_batches = 0
    for i in range(0, len(train_images), batch_size):
        batch_x = train_images[i:i+batch_size]
        batch_y = train_labels[i:i+batch_size]
        
        output = model(batch_x)
        loss = criterion(output, batch_y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
    
    avg_loss = epoch_loss / n_batches
    losses.append(avg_loss)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} | loss={avg_loss:.4f}")

# Save model
os.makedirs("models", exist_ok=True)
torch.save(model.state_dict(), "models/model.pth")

# Save training info
with open("models/train_info.json", "w") as f:
    json.dump({"final_loss": losses[-1], "num_epochs": num_epochs}, f)

print(f"Training complete. Final loss: {losses[-1]:.4f}")

In [ ]:
%%writefile evaluate.py
"""Stage 3: Evaluate model on validation set."""
import json
import numpy as np
import torch
import torch.nn as nn
import yaml

# Read config
with open("params.yaml") as f:
    config = yaml.safe_load(f)

# Load data
val_images = torch.from_numpy(np.load("data/processed/val_images.npy"))
val_labels = torch.from_numpy(np.load("data/processed/val_labels.npy"))

# Load model
num_classes = config["model"]["num_classes"]
features = config["model"]["features"]

model = nn.Sequential(
    nn.Conv2d(1, features, 3, padding=1), nn.ReLU(),
    nn.Conv2d(features, features * 2, 3, padding=1), nn.ReLU(),
    nn.Conv2d(features * 2, features, 3, padding=1), nn.ReLU(),
    nn.Conv2d(features, num_classes, 1),
)
model.load_state_dict(torch.load("models/model.pth", weights_only=True))
model.eval()

# Evaluate
with torch.no_grad():
    output = model(val_images)
    preds = output.argmax(dim=1)
    
    # Overall accuracy
    accuracy = (preds == val_labels).float().mean().item()
    
    # Per-class Dice (simplified)
    dice_scores = {}
    for c in range(num_classes):
        pred_c = (preds == c).float()
        label_c = (val_labels == c).float()
        intersection = (pred_c * label_c).sum()
        dice = (2 * intersection / (pred_c.sum() + label_c.sum() + 1e-8)).item()
        dice_scores[f"dice_class_{c}"] = round(dice, 4)
    
    mean_dice = np.mean(list(dice_scores.values()))

# Save metrics (DVC can track this file)
metrics = {
    "accuracy": round(accuracy, 4),
    "mean_dice": round(float(mean_dice), 4),
    **dice_scores
}

with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Evaluation: accuracy={accuracy:.4f}, mean_dice={mean_dice:.4f}")

## Create the Config File

DVC reads parameters from a `params.yaml` file. This is similar to your production YAML configs.

In [ ]:
%%writefile params.yaml
# Pipeline configuration (like spine_semantic_new_v8.yaml)

seed: 42

data:
  num_samples: 100
  val_ratio: 0.2

model:
  num_classes: 4
  features: 16

training:
  lr: 0.001
  batch_size: 8
  num_epochs: 20

## Define the Pipeline (`dvc.yaml`)

This is the core file. It describes the full workflow.

In [ ]:
%%writefile dvc.yaml
stages:

  prepare_data:
    cmd: python prepare_data.py
    deps:
      - prepare_data.py
    params:
      - data
      - seed
    outs:
      - data/processed

  train:
    cmd: python train.py
    deps:
      - train.py
      - data/processed
    params:
      - model
      - training
    outs:
      - models

  evaluate:
    cmd: python evaluate.py
    deps:
      - evaluate.py
      - models/model.pth
      - data/processed/val_images.npy
      - data/processed/val_labels.npy
    params:
      - model
    metrics:
      - metrics.json:
          cache: false

## Visualize the Pipeline

In [ ]:
# See the DAG (dependency graph)
!dvc dag

## Run the Pipeline

In [ ]:
# dvc repro = "reproduce" the pipeline
# Runs all stages in dependency order
!dvc repro

In [ ]:
# See the metrics
!dvc metrics show

In [ ]:
# Commit everything
!git add .
!git commit -m "First pipeline run: baseline"

## Change Config → Rerun Only What Changed

Let's increase training epochs and model size. DVC will figure out what to rerun.

In [ ]:
import yaml

# Update config
with open("params.yaml") as f:
    params = yaml.safe_load(f)

params["training"]["num_epochs"] = 40  # was 20
params["model"]["features"] = 32       # was 16

with open("params.yaml", "w") as f:
    yaml.dump(params, f, default_flow_style=False)

print("Updated params.yaml: epochs=40, features=32")

In [ ]:
# See what DVC thinks changed
!dvc status

In [ ]:
# Rerun — DVC skips prepare_data (data params didn't change)
!dvc repro

In [ ]:
# Compare metrics between runs
!dvc params diff
print("---")
!dvc metrics diff

In [ ]:
# Commit the improved run
!git add .
!git commit -m "Pipeline run 2: more epochs, bigger model"

## The Lock File (`dvc.lock`)

After running `dvc repro`, DVC creates `dvc.lock`. This captures the **exact** state of everything:

In [ ]:
!cat dvc.lock

The lock file contains hashes of every dependency and output. If anyone wants to reproduce this exact experiment, the lock file guarantees it.

This is like `package-lock.json` in npm — it pins exact versions.

## Connection to Your Production Code

The production training pipeline maps directly to DVC stages:

```yaml
# How a dvc.yaml for spine training would look:
stages:
  
  prepare_data:
    cmd: python -m src.dataset --config config/spine_semantic_new_v8.yaml
    deps:
      - src/dataset.py
      - data/data_splits/101225_s1ok/       # CSV splits
    params:
      - config/spine_semantic_new_v8.yaml:
          - dataset
    outs:
      - data/prepared/
  
  train_semantic:
    cmd: python -m src.training_semantic --config spine_semantic_new_v8.yaml
    deps:
      - src/training_semantic.py
      - data/prepared/
    params:
      - config/spine_semantic_new_v8.yaml:
          - model
          - training
    outs:
      - weights/training_spine_semantic_new_v8/
  
  evaluate_semantic:
    cmd: python -m src.testing_semantic --config spine_semantic_new_v8.yaml
    deps:
      - src/testing_semantic.py
      - weights/training_spine_semantic_new_v8/best_metric_model.pth
    metrics:
      - results/metrics_semantic.json:
          cache: false
```

Then:
- `dvc repro` → runs the full pipeline
- `dvc params diff` → shows what config changed between runs
- `dvc metrics diff` → shows if the model got better or worse
- `dvc.lock` → captures exact hashes (FDA audit trail)

## Summary

| Concept | What it does |
|---------|-------------|
| `dvc.yaml` | Defines pipeline stages (DAG) |
| `dvc repro` | Runs pipeline, skipping unchanged stages |
| `dvc dag` | Visualizes the dependency graph |
| `dvc.lock` | Pins exact versions of everything |
| `dvc params diff` | Compares config between commits |
| `dvc metrics diff` | Compares results between commits |
| `dvc status` | Shows what's out of date |

**Key takeaway:** DVC pipelines make ML experiments reproducible. Change a config, run `dvc repro`, and DVC handles the rest. The lock file gives you a complete audit trail.

**Next:** Putting it all together — DVC + MLflow in one workflow.